# Job run — CO-Bench — Graph Colouring

Evolve a `solve(**instance)` graph-colouring algorithm; scored by CO-Bench's per-task evaluator (higher normalized score = better).

Two **independent** cluster jobs, each with its own *submit / status / kill* cells:

1. **Normal run** — `scripts/run_evolution.sh --seed --config-name config_co-bench experiments.co_bench.CO_BENCH_TASK=GRAPH_COLOURING` (starts Ollama, seeds gen 0, evolves to the configured stop criteria).
2. **Baseline** — `scripts/run_shinka_baseline.sh --config-name baselines/co-bench experiments.co_bench.CO_BENCH_TASK=GRAPH_COLOURING` (ShinkaEvolve over the same evaluator + local Ollama).

Canonical Comet run-name (set by the repo config, **unchanged**): `cobench-graph_colouring`. Run *Common config* first, then either job group.

> **CO-Bench data:** both jobs chain `scripts/bootstrap_cobench.sh` first (clone the CO-Bench checkout + download the dataset into `./data/co-bench`). It is idempotent — re-runs skip work that already exists.

## Common config

In [ ]:
import client_lib

# ---------------------------------------------------------------------------
# Cluster + repo paths
# ---------------------------------------------------------------------------
WORK_DIR = "/home/jovyan/echimbulatov/fork_afedorov/constant_repos/new-optimizer-finding"
BASE_IMAGE = "cr.ai.cloud.ru/2754eb6e-ae19-4123-87ce-06ec3cc96500/job-latentdiffusion:flash-clear"

# ---------------------------------------------------------------------------
# Hardware - a100.{N_GPUS}gpu SKU in the A100-MT region (verbatim from the
# canonical job-run notebook on this cluster). The organism-first evolution
# loop is a SINGLE-PROCESS orchestrator; the 8 GPUs host the local Ollama
# model instances that scripts/run_evolution.sh + scripts/lib_runtime.sh
# start (gemma 4 31B + qwen 3.5 across the 8 GPUs - see the experiment config).
# ---------------------------------------------------------------------------
N_NODES = 1
N_GPUS = 8
instance_cpu_count = 8
instance_addition = (
    f".{N_GPUS * instance_cpu_count}C.{N_GPUS * 243}G"
    if instance_cpu_count == 8 else ""
)
INSTANCE_TYPE = f"a100.{N_GPUS}gpu{instance_addition}"
REGION = "A100-MT"

_total_gpus = N_NODES * N_GPUS
print(f"Instance: {INSTANCE_TYPE}  (region={REGION})")
print(f"Total GPUs: {_total_gpus}  (= {N_NODES} nodes x {N_GPUS} GPUs)")

# ---------------------------------------------------------------------------
# Experiment identity. EXP_BASE is the human label used in the job description
# only. The system-visible run identity (the Comet run-name) is owned by the
# repo's Hydra config (cfg.comet.run_name) and left at its canonical default:
#   cobench-graph_colouring
# ---------------------------------------------------------------------------
EXP_BASE = "cobench-graph_colouring"
RUN_CONFIG = "config_co-bench"             # normal evolution-run preset
BASELINE_CONFIG = "baselines/co-bench"   # ShinkaEvolve baseline preset
OVERRIDES = " experiments.co_bench.CO_BENCH_TASK=GRAPH_COLOURING"               # hydra overrides appended to BOTH jobs
NEEDS_COBENCH_BOOTSTRAP = True   # CO-Bench needs the dataset + checkout first

# ---------------------------------------------------------------------------
# Common job env - cluster scaffolding + Comet creds. Same Comet api_key and
# workspace as the canonical notebook (the repo's Hydra config bakes in the
# same api_key too); COMET_PROJECT points at this project's Comet project.
# COMET_RUN_NAME is intentionally NOT set, so the config's canonical per-task
# run-name is preserved.
# ---------------------------------------------------------------------------
common_env = {
    "PROJECT_ROOT": WORK_DIR,
    "PYTHONNOUSERSITE": 1,
    "PIP_USER": "no",
    "NCCL_DEBUG": "INFO",
    "NCCL_IB_TIMEOUT": 23,
    "NCCL_IB_RETRY_CNT": 5,
    "TORCH_NCCL_HEARTBEAT_TIMEOUT_SEC": 3600,
    "MLS_JOB_REGION_NAME": REGION,
    "MLS_JOB_TOTAL_GPU": _total_gpus,
    "CLEARML_CONFIG_FILE": "/home/jovyan/inkoziev/myclearml.conf",
    "COMET_API_KEY": "RrClhd4FveFQKO4qLo4jBjrKu",
    "COMET_WORKSPACE": "dont4rootme",
    "COMET_PROJECT": "new-optimizer-search",
    "COMET_MODE": "online",
    "COMET_LOGGING_CONSOLE": "true",
}

## Job 1 — Normal evolution run

In [ ]:
# ---------------------------------------------------------------------------
# Normal training run = the organism-first evolution loop.
# scripts/run_evolution.sh:
#   * starts/refreshes the local Ollama instances declared in the config
#     (scripts/lib_runtime.sh) on the node GPUs,
#   * with --seed, bootstraps the generation-0 population if missing,
#   * runs seeding + evolution to the configured stop criteria
#     (max_generations / max_organism_creations / per-model token budget).
# ---------------------------------------------------------------------------
_bootstrap = f"bash {WORK_DIR}/scripts/bootstrap_cobench.sh && " if NEEDS_COBENCH_BOOTSTRAP else ""
run_script = (
    f"cd {WORK_DIR} && {_bootstrap}"
    f"bash scripts/run_evolution.sh --seed --config-name {RUN_CONFIG}{OVERRIDES}"
)
EXP_NAME_RUN = f"{EXP_BASE}-run"
print(f"[{EXP_NAME_RUN}]")
print(run_script)

In [ ]:
run_env = dict(common_env)

# `#ID0137 #rnd` are the user-quota / priority-category tags the cluster
# scheduler needs (verbatim from the canonical job-run notebook); without them
# the job lands in a default bucket with a short wall-time limit.
run_job = client_lib.Job(
    job_desc=f"echimbulatov | {EXP_NAME_RUN} #ID0137 #rnd",
    queue_name="diff",
    base_image=BASE_IMAGE,
    script=run_script,
    n_workers=N_NODES,
    instance_type=INSTANCE_TYPE,
    type="pytorch2",
    preflight_check=True,
    env_variables=run_env,
    region=REGION,
    flags={},
    priority_class="high",
)
run_job.submit()

In [ ]:
while run_job.status() == "Job status=Pending":
    run_job.status()
run_job.logs()

In [ ]:
# run_job.kill()

## Job 2 — ShinkaEvolve baseline

In [ ]:
# ---------------------------------------------------------------------------
# Baseline = ShinkaEvolve over the SAME task evaluator + local Ollama models.
# scripts/run_shinka_baseline.sh shares the Ollama lifecycle with the run above
# and invokes src.baselines.shinka.run (ShinkaEvolve dataclasses + our
# evaluator). Writes its per-program DB under shinka_runs/.
# ---------------------------------------------------------------------------
_bootstrap = f"bash {WORK_DIR}/scripts/bootstrap_cobench.sh && " if NEEDS_COBENCH_BOOTSTRAP else ""
baseline_script = (
    f"cd {WORK_DIR} && {_bootstrap}"
    f"bash scripts/run_shinka_baseline.sh --config-name {BASELINE_CONFIG}{OVERRIDES}"
)
EXP_NAME_BASELINE = f"{EXP_BASE}-baseline"
print(f"[{EXP_NAME_BASELINE}]")
print(baseline_script)

In [ ]:
baseline_env = dict(common_env)

baseline_job = client_lib.Job(
    job_desc=f"echimbulatov | {EXP_NAME_BASELINE} #ID0137 #rnd",
    queue_name="diff",
    base_image=BASE_IMAGE,
    script=baseline_script,
    n_workers=N_NODES,
    instance_type=INSTANCE_TYPE,
    type="pytorch2",
    preflight_check=True,
    env_variables=baseline_env,
    region=REGION,
    flags={},
    priority_class="high",
)
baseline_job.submit()

In [ ]:
while baseline_job.status() == "Job status=Pending":
    baseline_job.status()
baseline_job.logs()

In [ ]:
# baseline_job.kill()